In [1]:
print("Hello")

Hello


In [2]:
import pandas as pd
rules_list = [
    {
        "rule_id": 1,
        "category": "Pricing",
        "short description": "Check pricing keyword",
        "long description": "Verifies that the content contains the keyword 'pricing' (case-insensitive) to ensure pricing details are provided.",
        "active": 1
    },
    {
        "rule_id": 2,
        "category": "Rates",
        "short description": "Check rates keyword",
        "long description": "Checks the content for the keyword 'rates' to confirm rate information is included.",
        "active": 1
    },
    {
        "rule_id": 3,
        "category": "Sales",
        "short description": "Check sales keyword",
        "long description": "Validates that the file includes 'sales' to ensure proper sales details are available.",
        "active": 1
    },
    {
        "rule_id": 4,
        "category": "Revenue",
        "short description": "Check revenue keyword",
        "long description": "Ensures the occurrence of 'revenue' in the content to verify revenue reporting.",
        "active": 1
    },
    {
        "rule_id": 5,
        "category": "Invoice",
        "short description": "Check invoice keyword",
        "long description": "Verifies that 'invoice' is present to check for invoice details.",
        "active": 1
    },
    {
        "rule_id": 6,
        "category": "Cost",
        "short description": "Check cost keyword",
        "long description": "Checks for the keyword 'cost' in the file to ensure that cost-related metrics are mentioned.",
        "active": 1
    },
    {
        "rule_id": 7,
        "category": "Discount",
        "short description": "Check discount keyword",
        "long description": "Validates that the content contains 'discount', indicating that discount information is available.",
        "active": 1
    },
    {
        "rule_id": 8,
        "category": "Margin",
        "short description": "Check margin keyword",
        "long description": "Ensures that the file contains 'margin' to confirm margin details are provided.",
        "active": 1
    },
    {
        "rule_id": 9,
        "category": "Tax",
        "short description": "Check tax keyword",
        "long description": "Verifies that 'tax' is present in the content, indicating that tax details are included.",
        "active": 1
    },
    {
        "rule_id": 10,
        "category": "Total",
        "short description": "Check total keyword",
        "long description": "Checks for the keyword 'total' to ensure that total amounts are referenced in the file.",
        "active": 1
    },
]

compliance_df = pd.DataFrame(rules_list)
compliance_df.head()


,rule_id,category,short description,long description,active
0,1,Pricing,Check pricing keyword,Verifies that the content contains the keyword...,1
1,2,Rates,Check rates keyword,Checks the content for the keyword 'rates' to ...,1
2,3,Sales,Check sales keyword,Validates that the file includes 'sales' to en...,1
3,4,Revenue,Check revenue keyword,Ensures the occurrence of 'revenue' in the con...,1
4,5,Invoice,Check invoice keyword,Verifies that 'invoice' is present to check fo...,1


In [4]:
import pandas as pd

# Define compliance rules in a structured format
compliance_rules = [
    ["P001", "Pricing & Invoicing", "Invoice/Billing Mentions", 
     "Should not contain references to invoicing or billing details, including terms like 'invoice number, amount due, payment terms, billing cycle'.", 1],
    ["P002", "Pricing & Invoicing", "Monetary Transactions", 
     "Should not mention monetary transactions associated with pricing, fees, or payments, including currency symbols (e.g., '$', 'USD', 'EUR').", 1],
    ["P003", "Pricing & Invoicing", "Pricing Structure Disclosure", 
     "Should not disclose pricing structures or rate breakdowns, including 'product/service pricing tables, total cost calculations, or unit pricing details'.", 1],
    ["P004", "Pricing & Invoicing", "Discounts & Tax Mentions", 
     "Should not include references to discounts, taxation, or additional fees, such as 'rebates, VAT/GST, tax calculations, surcharges, or late fees'.", 1],
    ["P005", "Pricing & Invoicing", "Contractual Payment Terms", 
     "Should not contain contractual payment obligations or financial commitments, including terms like 'net 30, advance payment, installment terms, or late payment penalties'.", 1],
    ["C001", "Confidential Market Intelligence", "Competitor Mentions & Comparisons", 
     "Should not mention competitor names or engage in direct competitive comparisons, including 'product benchmarking, competitive analysis, or price-matching strategies'.", 1],
    ["C002", "Confidential Market Intelligence", "Market Share & Growth Insights", 
     "Should not disclose market share insights or growth projections, such as 'industry positioning, revenue forecasts, or business expansion plans'.", 1],
    ["C003", "Confidential Market Intelligence", "Business Strategy & Roadmap", 
     "Should not reveal confidential business strategy or roadmap details, including 'internal product development plans, M&A activities, or go-to-market strategies'.", 1],
    ["C004", "Confidential Market Intelligence", "Competitive Pricing & Cost Analysis", 
     "Should not contain references to competitive pricing models or cost structures, including 'pricing intelligence, competitor rate analysis, or internal pricing strategies'.", 1],
    ["C005", "Confidential Market Intelligence", "Proprietary Market Research & Intelligence", 
     "Should not include proprietary market research or non-public industry data, such as 'internal research reports, customer sentiment analysis, or exclusive market intelligence'.", 1]
]

# Create a DataFrame
df = pd.DataFrame(compliance_rules, columns=["Rule ID", "Category", "Short Description", "Full Description", "Active"])

# Save as CSV
csv_file_path = "compliance_rules.csv"
df.to_csv(csv_file_path, index=False)

df.head()

,Rule ID,Category,Short Description,Full Description,Active
0,P001,Pricing & Invoicing,Invoice/Billing Mentions,Should not contain references to invoicing or ...,1
1,P002,Pricing & Invoicing,Monetary Transactions,Should not mention monetary transactions assoc...,1
2,P003,Pricing & Invoicing,Pricing Structure Disclosure,Should not disclose pricing structures or rate...,1
3,P004,Pricing & Invoicing,Discounts & Tax Mentions,"Should not include references to discounts, ta...",1
4,P005,Pricing & Invoicing,Contractual Payment Terms,Should not contain contractual payment obligat...,1


In [ ]:
import os
import json
import argparse
from typing import List, Dict, Any, Optional, Tuple
import PyPDF2
import docx
import tiktoken
import openai
import logging
from tqdm import tqdm

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

class DocumentComplianceChecker:
    """A class to check document compliance using OpenAI's GPT model."""
    
    def __init__(self, api_key: str, model: str = "gpt-4o-mini"):
        """
        Initialize the Document Compliance Checker.
        
        Args:
            api_key: OpenAI API key
            model: OpenAI model to use (default: gpt-4o-mini)
        """
        self.api_key = api_key
        self.model = model
        openai.api_key = api_key
        
        # GPT-4o-mini context window is 128K tokens
        self.max_tokens = 128000 if "gpt-4" in model else 16385
        # Reserve 1000 tokens for the prompt and 2000 for the response
        self.max_chunk_tokens = self.max_tokens - 3000
        
        # Initialize tokenizer for the model
        self.tokenizer = tiktoken.encoding_for_model(model)
        
    def extract_text_from_pdf(self, file_path: str) -> str:
        """
        Extract text from a PDF file.
        
        Args:
            file_path: Path to the PDF file
            
        Returns:
            The extracted text
        """
        logger.info(f"Extracting text from PDF: {file_path}")
        text = ""
        
        try:
            with open(file_path, 'rb') as file:
                pdf_reader = PyPDF2.PdfReader(file)
                for page_num in range(len(pdf_reader.pages)):
                    page = pdf_reader.pages[page_num]
                    text += page.extract_text() + "\n\n"
                    
            logger.info(f"Successfully extracted text from {len(pdf_reader.pages)} pages")
            return text
        except Exception as e:
            logger.error(f"Error extracting text from PDF: {e}")
            raise
            
    def extract_text_from_docx(self, file_path: str) -> str:
        """
        Extract text from a DOCX file.
        
        Args:
            file_path: Path to the DOCX file
            
        Returns:
            The extracted text
        """
        logger.info(f"Extracting text from DOCX: {file_path}")
        text = ""
        
        try:
            doc = docx.Document(file_path)
            for para in doc.paragraphs:
                text += para.text + "\n"
                
            logger.info(f"Successfully extracted text from DOCX with {len(doc.paragraphs)} paragraphs")
            return text
        except Exception as e:
            logger.error(f"Error extracting text from DOCX: {e}")
            raise
            
    def extract_text(self, file_path: str) -> str:
        """
        Extract text from a file based on its extension.
        
        Args:
            file_path: Path to the file
            
        Returns:
            The extracted text
        """
        file_extension = os.path.splitext(file_path)[1].lower()
        
        if file_extension == '.pdf':
            return self.extract_text_from_pdf(file_path)
        elif file_extension == '.docx':
            return self.extract_text_from_docx(file_path)
        else:
            raise ValueError(f"Unsupported file type: {file_extension}. Only PDF and DOCX are supported.")
            
    def split_text_into_chunks(self, text: str) -> List[str]:
        """
        Split text into chunks that fit within the model's context window.
        
        Args:
            text: The text to split
            
        Returns:
            A list of text chunks
        """
        tokens = self.tokenizer.encode(text)
        chunks = []
        current_chunk = []
        current_chunk_length = 0
        
        for token in tokens:
            if current_chunk_length + 1 <= self.max_chunk_tokens:
                current_chunk.append(token)
                current_chunk_length += 1
            else:
                chunks.append(self.tokenizer.decode(current_chunk))
                current_chunk = [token]
                current_chunk_length = 1
                
        if current_chunk:
            chunks.append(self.tokenizer.decode(current_chunk))
            
        logger.info(f"Split text into {len(chunks)} chunks")
        return chunks
        
    def analyze_chunk(self, chunk: str, chunk_number: int, total_chunks: int, compliance_rules: List[str]) -> Dict[str, Any]:
        """
        Analyze a chunk of text for compliance using GPT.
        
        Args:
            chunk: The text chunk to analyze
            chunk_number: The current chunk number
            total_chunks: The total number of chunks
            compliance_rules: The compliance rules to check against
            
        Returns:
            The compliance analysis results for the chunk
        """
        rules_text = "\n".join([f"{i+1}. {rule}" for i, rule in enumerate(compliance_rules)])
        
        prompt = f"""
You are an AI document compliance checker. Analyze the following document text for compliance with the rules listed below.
This is chunk {chunk_number} of {total_chunks}.

COMPLIANCE RULES:
{rules_text}

YOUR TASK:
1. Analyze the document text for compliance with each rule.
2. For each rule, determine if the document is compliant, non-compliant, or if compliance cannot be determined from this chunk.
3. For non-compliant items, provide specific examples from the text and explain why they violate the rule.
4. Format your response as a JSON object with the following structure:
   {{
     "chunk_number": {chunk_number},
     "results": [
       {{
         "rule_number": 1,
         "rule_text": "Rule 1 text",
         "status": "compliant" | "non_compliant" | "undetermined",
         "evidence": "Evidence or examples from the text (only for non-compliant items)",
         "explanation": "Explanation of why it violates the rule (only for non-compliant items)"
       }},
       ...
     ]
   }}

DOCUMENT TEXT:
{chunk}

Respond with only the JSON object. Do not include any other text in your response.
"""

        try:
            response = openai.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                response_format={"type": "json_object"}
            )
            
            analysis = json.loads(response.choices[0].message.content)
            logger.info(f"Successfully analyzed chunk {chunk_number}/{total_chunks}")
            return analysis
        except Exception as e:
            logger.error(f"Error analyzing chunk {chunk_number}: {e}")
            return {
                "chunk_number": chunk_number,
                "results": [],
                "error": str(e)
            }
            
    def analyze_document(self, file_path: str, compliance_rules: List[str]) -> Dict[str, Any]:
        """
        Analyze a document for compliance.
        
        Args:
            file_path: Path to the document
            compliance_rules: List of compliance rules to check against
            
        Returns:
            The compliance analysis report
        """
        logger.info(f"Starting compliance analysis for {file_path}")
        
        # Extract text from document
        text = self.extract_text(file_path)
        
        # Split text into chunks
        chunks = self.split_text_into_chunks(text)
        
        # Analyze each chunk
        chunk_results = []
        for i, chunk in enumerate(tqdm(chunks, desc="Analyzing chunks")):
            result = self.analyze_chunk(chunk, i+1, len(chunks), compliance_rules)
            chunk_results.append(result)
            
        # Compile the final report
        report = self.compile_report(file_path, compliance_rules, chunk_results)
        return report
        
    def compile_report(self, file_path: str, compliance_rules: List[str], chunk_results: List[Dict[str, Any]]) -> Dict[str, Any]:
        """
        Compile the final compliance report from chunk results.
        
        Args:
            file_path: Path to the document
            compliance_rules: List of compliance rules that were checked
            chunk_results: Results from analyzing each chunk
            
        Returns:
            The compiled compliance report
        """
        logger.info("Compiling final compliance report")
        
        rule_statuses = {}
        rule_evidence = {}
        
        # Initialize rule statuses
        for i, rule in enumerate(compliance_rules):
            rule_number = i + 1
            rule_statuses[rule_number] = "undetermined"
            rule_evidence[rule_number] = []
            
        # Process chunk results
        for chunk_result in chunk_results:
            if "error" in chunk_result:
                continue
                
            for result in chunk_result.get("results", []):
                rule_number = result.get("rule_number")
                status = result.get("status")
                
                if rule_number not in rule_statuses:
                    continue
                    
                # If any chunk shows non-compliance, the rule is non-compliant
                if status == "non_compliant":
                    rule_statuses[rule_number] = "non_compliant"
                    evidence = {
                        "chunk": chunk_result.get("chunk_number"),
                        "evidence": result.get("evidence", ""),
                        "explanation": result.get("explanation", "")
                    }
                    rule_evidence[rule_number].append(evidence)
                # If status is compliant and current status is undetermined, update to compliant
                elif status == "compliant" and rule_statuses[rule_number] == "undetermined":
                    rule_statuses[rule_number] = "compliant"
                    
        # Compile final report
        report = {
            "document": os.path.basename(file_path),
            "compliance_summary": {
                "total_rules": len(compliance_rules),
                "compliant_rules": sum(1 for status in rule_statuses.values() if status == "compliant"),
                "non_compliant_rules": sum(1 for status in rule_statuses.values() if status == "non_compliant"),
                "undetermined_rules": sum(1 for status in rule_statuses.values() if status == "undetermined")
            },
            "rule_results": []
        }
        
        for i, rule in enumerate(compliance_rules):
            rule_number = i + 1
            rule_result = {
                "rule_number": rule_number,
                "rule_text": rule,
                "status": rule_statuses[rule_number],
                "evidence": rule_evidence[rule_number] if rule_statuses[rule_number] == "non_compliant" else []
            }
            report["rule_results"].append(rule_result)
            
        return report
        
    def save_report(self, report: Dict[str, Any], output_path: str) -> None:
        """
        Save the compliance report to a file.
        
        Args:
            report: The compliance report
            output_path: Path to save the report
        """
        try:
            with open(output_path, 'w') as f:
                json.dump(report, f, indent=2)
            logger.info(f"Report saved to {output_path}")
        except Exception as e:
            logger.error(f"Error saving report: {e}")
            raise

def main():
    """Main function to run the document compliance checker."""
    parser = argparse.ArgumentParser(description='AI-Based Document Compliance Checker')
    parser.add_argument('--file', required=True, help='Path to the document file (PDF or DOCX)')
    parser.add_argument('--rules', required=True, help='Path to JSON file containing compliance rules')
    parser.add_argument('--output', default='compliance_report.json', help='Path to save the compliance report')
    parser.add_argument('--api-key', required=True, help='OpenAI API key')
    parser.add_argument('--model', default='gpt-4o-mini', help='OpenAI model to use')
    
    args = parser.parse_args()
    
    try:
        # Load compliance rules
        with open(args.rules, 'r') as f:
            compliance_rules = json.load(f)
            
        # Initialize checker
        checker = DocumentComplianceChecker(api_key=args.api_key, model=args.model)
        
        # Analyze document
        report = checker.analyze_document(args.file, compliance_rules)
        
        # Save report
        checker.save_report(report, args.output)
        
        logger.info("Compliance analysis completed successfully")
        
    except Exception as e:
        logger.error(f"Error in compliance analysis: {e}", exc_info=True)
        return 1
        
    return 0

if __name__ == "__main__":
    exit(main())